In [2]:

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"


q1vec = model.encode(q1)
q2vec = model.encode(q2)

print(q1vec.shape)

print (q1vec.dot(q2vec))
q3 = 'Can I still join the course after the start date?'
q3vec = model.encode(q3)

d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dvec = model.encode(d)


r = q1vec.dot(dvec)
print(r)

q4 = 'How to install Docker on Windows?'
q4vec = model.encode(q4)

v4 = q4vec.dot(dvec)
print(f"d: {d} dot q4:{q4} = {v4}")

(384,)
0.6227728
0.39572883
d: You don't need to register. You're accepted. You can also just start learning and submitting homework without registering. dot q4:How to install Docker on Windows? = 0.01973043382167816


In [4]:
from ingest import load_faq_data

documents = load_faq_data()

texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

documents[0]


from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [5]:

import numpy as np
X = np.array(vectors)
print(X.shape)

scores = X.dot(q1vec)

idx = np.argmax(scores)
idx, scores[idx]

print(f"Question: {q1}")
print(documents[idx])

top5 = np.argsort(scores)[-5:]
scores[top5]

top5 = np.argsort(-scores)[:5]
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

(1350, 384)
Question: I just discovered the course, can I still join?
{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'doc_id': '74eb249bbf'}
0.831779
{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'doc_id': '74eb249bbf'}

0.6845695
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still t

In [6]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)


results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)


In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

query = "I just found out about the program, can I still sign up?"
ans = assistant.rag(query)
print(ans)

Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


In [8]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )
    

vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)
    
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. You can start learning and submit homework while the form is open. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

vs_index.fit(vectors, documents)


In [11]:

query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

print(results)

[{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'doc_id': '74eb249bbf'}, {'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.', 'doc_id': '41aabbd7c5'}, {'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions